# 03. 마피아 게임 실행 (GameEngine 버전)

**목적**: 순수 파이썬 `GameEngine`으로 마피아 한 판을 진행한다. 두 가지 모드를 지원한다.

| 모드 | 구성 | 설명 |
|---|---|---|
| 관전 모드 | AI 8명 | 사람은 구경만 한다. 마피아의 밤 대화/의사/경찰 선택과 **AI 마다의 사고 과정**도 보인다 |
| 참여 모드 | 사람 1 + AI 7 | 사람 정체 정보만 보인다. 발언, 투표를 직접 입력한다. `?질문`으로 사회자에게 규칙 문의 |

`GameEngine`의 역할 분담 (**LLM은 연기, 진실은 코드가 지킨다**)

- 엔진(코드): 밤 처리, 사망 발표, 투표 집계, 동점 재투표, 승리 판정, 정보 분리
- 컨트롤러(LLM / 사람): 발언과 선택만 담당
- 게임이 끝나면 `reveal_report()`로 전원의 역할과 속마음을 공개하고, `save_log()`로 JSON 저장

진행: 밤 1 -> 낮 1(토론) -> 투표 -> 밤 2 -> ... (마피아 전멸 시 시민 승, 마피아 수 >= 시민 수 시 마피아 승, 10일 초과 무승부)

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
C:\Users\my\dev\mafia-agent
LLM: qwen3:8b / 임베딩: bge-m3


In [2]:
# ===== 1. 준비 =====
import random
import time

import rag
import mafia_core as core
import mafia_agents as agents

vs = rag.load_db()
print("===== 벡터 DB 로드 완료 (Chunk %d개) =====" % vs._collection.count())

===== 벡터 DB 로드 완료 (Chunk 23개) =====


## 1. 관전 모드: AI 8명 + 사고 과정 보기

`build_controllers()`는 플레이어마다 역할에 맞는 도구와 캐릭터 페르소나를 가진 `LLMPlayer`를 만들어 준다.
**한 판에 수 분이 걸린다.** `SEED`를 바꾸면 다른 역할 배정이 나온다.

`SHOW_THINKING = True`이면 AI가 결정할 때마다 **사고 과정** 상자가 함께 나온다. 챗봇(질문 -> 답변)과 달리,
각 AI가 *자기 정보만 받고 -> 룰북을 검색하고 -> 도구를 호출해 -> 결정하는* 에이전트라는 것을 눈으로 확인할 수 있다.

| 상자 안의 줄 | 의미 |
|---|---|
| 받은 정보(PlayerView) / 나만 아는 정보 | 코드가 골라 준 그 AI 전용 정보 (다른 사람의 직업은 없음) |
| RAG 검색 | 메타데이터 필터로 읽은 룰북 페이지 (내 역할 전략, 내 캐릭터 프로필) |
| 도구 호출 / 도구 응답 | `create_agent`가 고른 도구(`choose_kill_target`, `investigate`, `protect`, `vote`)와 인자, 도구의 후보 검사 결과 |
| 전략 판단(코드) / 몰아가기 감지(코드) | 코드가 상황을 계산해 내린 지시 (예: 경찰이 정체를 밝힐 시점) |
| 안전장치 작동 / 후처리(코드) | 반복, 자기 이름 3인칭, 없는 밤 행동 언급 등을 감지해 재생성하거나 고친 기록 |
| 속마음 / 의심도 | 공개 발언과 따로 남는 비공개 추리 (게임이 끝나면 공개) |

> 사고 과정은 다른 사람의 비밀을 보여 주므로 **관전 모드에서만** 켜진다. 사람이 참여하면 자동으로 꺼진다.

In [3]:
# ===== 2. 관전 모드 실행 =====
SEED = 11
SHOW_THINKING = True   # AI 사고 과정(받은 정보 -> RAG -> 도구 호출 -> 안전장치 -> 결정) 출력. 결과만 보려면 False

state = core.new_game(list(config.CHARACTERS), None, seed=SEED)
controllers = agents.build_controllers(state, vs, show_thinking=SHOW_THINKING)
engine = core.GameEngine(state, controllers)   # on_event 기본값: 한국어로 출력

t0 = time.time()
winner = engine.run()
elapsed = time.time() - t0

print("\n===== 게임 종료 =====")
print("승자:", winner, "| 진행 일수:", state.day, "| 소요 시간: %.0f초" % elapsed)


===== 마피아 게임 시작 (8명) =====
참가자: 강태호, 윤서진, 박준수, 최민지, 정도현, 한소율, 오지환, 임채원
  [비밀 · 마피아 강태호] 당신의 직업: 마피아
  [비밀 · 마피아 강태호] 동료 마피아: 윤서진
  [비밀 · 마피아 윤서진] 당신의 직업: 마피아
  [비밀 · 마피아 윤서진] 동료 마피아: 강태호
  [비밀 · 경찰 박준수] 당신의 직업: 경찰
  [비밀 · 시민 최민지] 당신의 직업: 시민
  [비밀 · 의사 정도현] 당신의 직업: 의사
  [비밀 · 시민 한소율] 당신의 직업: 시민
  [비밀 · 시민 오지환] 당신의 직업: 시민
  [비밀 · 시민 임채원] 당신의 직업: 시민

===== 1일차 밤 =====
    ┌─ 사고 과정 · 마피아 강태호 · 밤 처치
    │ 받은 정보(PlayerView): 생존 8명 · 공개 기록 0줄 · 지난 속마음 0개 · 후보 6명
    │ 나만 아는 정보: 동료 마피아: 윤서진
    │ RAG 검색: p.5 마피아 전략 가이드 (role=mafia 필터)
    │ 도구 호출: choose_kill_target(target='박준수')
    │   └ 도구 응답: 박준수 선택 완료. 한 줄로 마무리하세요.
    │ 이유: 박준수는 경찰/의사로 보이는 사람으로, 논리적으로 추리해 주목받는 사람으로 판단되며, 시민팀에 위협이 될 수 있는 인물입니다.
    └─ 결정: 박준수 (에이전트가 도구를 호출해 결정)
  [비밀 · 마피아 강태호] 박준수를 노립니다 (이유: 박준수는 경찰/의사로 보이는 사람으로, 논리적으로 추리해 주목받는 사람으로 판단되며, 시민팀에 위협이 될 수 있는 인물입니다.) [제안]
    ┌─ 사고 과정 · 마피아 윤서진 · 밤 처치
    │ 받은 정보(PlayerView): 생존 8명 · 공개 기록 0줄 · 지난 속마음 0개 · 후보 6명
    │ 나만 아는 정보: 동료 마피아: 강태호
    │ 동료 마피아의 제안: 박준수
    │ RAG 검색:

## 2. 게임 종료 후: 속마음 공개, 로그 저장, 통계

In [4]:
# ===== 3. 결과 공개 =====
print(core.reveal_report(state))

===== 게임 결과: 시민팀 승리 (2일차) =====

[역할 공개]
  강태호 : 마피아 (사망)
  윤서진 : 마피아 (사망)
  박준수 : 경찰 (사망)
  최민지 : 시민 (사망)
  정도현 : 의사 (생존)
  한소율 : 시민 (생존)
  오지환 : 시민 (생존)
  임채원 : 시민 (생존)

[밤 기록]
  1일차: 처치 박준수, 보호 정도현, 사망 박준수, 경찰 조사 강태호(마피아)
  2일차: 처치 최민지, 보호 윤서진, 사망 최민지

[속마음 요약]
  <강태호 / 마피아>
    1일차 night: [밤 처치 선택] 박준수: 박준수는 경찰/의사로 보이는 사람으로, 논리적으로 추리해 주목받는 사람으로 판단되며, 시민팀에 위협이 될 수 있는 인...
    1일차 day: 윤서진은 내가 의심하지 않아. 최민지가 박준수 죽이자고 했는데, 그 말이 진짜야. 정도현은 말을 안 해. 의심도는 최민지 80, 정도현 50, ... 의심: 최민지 80, 정도현 50
    1일차 vote: [투표] 최민지: 최민지가 박준수 죽이자고 했고, 이는 마피아의 행동일 가능성이 높습니다.
  <윤서진 / 마피아>
    1일차 day: 처치된 박준수는 경찰이었고, 마피아는 밤에 움직였습니다. 강태호는 동료이기 때문에 의심하지 않지만, 그의 발언이 달라졌다는 점은 주목할 만합니다... 의심: 최민지 25, 임채원 20
    1일차 vote: [투표] 최민지: 최민지가 강태호를 의심하고 있는 점, 그리고 그녀의 발언이 다른 사람과 달라졌다는 점을 고려하면, 그녀가 마피아일 가능성이 높...
    2일차 night: [밤 처치 선택] 최민지: 최민지가 강태호를 의심하고 있으며, 그녀의 발언이 달라졌다는 점에서 마피아일 가능성이 높습니다. 이로 인해 시민팀에 ...
    2일차 day: 강태호는 이미 처형되었으므로 더 이상 의심할 필요가 없습니다. 최민지가 사망한 후, 윤서진은 그를 보호하려는 태도를 보였으나, 이는 단순한 시민... 의심: 임채원 60, 오지환 4

In [5]:
# ===== 4. 로그 저장 + 통계 =====
log_path = core.save_log(state)
print("===== 로그 저장 완료 =====")
print("저장 위치:", log_path)

print("\n===== 안전장치 통계 =====")
print("AI 결정 요청 수 :", state.stats["llm_calls"])   # 발언/투표/밤 행동 요청 수 (실제 LLM 호출은 더 많음)
print("재시도 횟수      :", state.stats["retries"])
print("fallback 횟수    :", state.stats["fallbacks"])
print("정체 노출 차단   :", state.stats["leak_blocked"])
print("일차당 평균 소요 : %.0f초" % (elapsed / max(state.day, 1)))

===== 로그 저장 완료 =====
저장 위치: C:\Users\my\dev\mafia-agent\logs\mafia_20261124_004435_seed11.json

===== 안전장치 통계 =====
AI 결정 요청 수 : 30
재시도 횟수      : 0
fallback 횟수    : 0
정체 노출 차단   : 0
일차당 평균 소요 : 292초


## 3. 시연용: 녹화된 판 다시 보기

실제 LLM 한 판은 수 분이 걸리고 결과도 매번 다르다. 발표 시간에 맞추기 위해,
같은 코드(사고 과정 보기 켜짐)로 **미리 진행한 실제 한 판의 출력**을 `data/demo/` 에 저장해 두었다.
아래 셀은 그 기록을 다시 출력할 뿐이며 LLM을 호출하지 않는다. (실시간 진행은 위의 1번 셀)

In [6]:
# ===== 5. 녹화된 판 재생 (LLM 호출 없음) =====
import json

DEMO_TXT = config.DATA_DIR / "demo" / "demo_game.txt"     # 진행 화면 그대로
DEMO_LOG = config.DATA_DIR / "demo" / "demo_game.json"    # save_log() 결과 (역할, 속마음 포함)
DELAY = 0.05   # 줄 사이 간격(초). 0 이면 한 번에 출력

meta = json.loads(DEMO_LOG.read_text(encoding="utf-8"))
print("===== 녹화된 판: seed %s, 승자 %s, %s일차에 종료 =====\n" % (meta["seed"], meta["winner"], meta["day"]))
for line in DEMO_TXT.read_text(encoding="utf-8").splitlines():
    print(line, flush=True)
    if DELAY:
        time.sleep(DELAY)

===== 녹화된 판: seed 11, 승자 citizen, 2일차에 종료 =====

# 03 노트북 관전 모드(SEED=11, SHOW_THINKING=True)를 2026-11-24 실제 qwen3:8b 로 실행한 출력 그대로

===== 마피아 게임 시작 (8명) =====
참가자: 강태호, 윤서진, 박준수, 최민지, 정도현, 한소율, 오지환, 임채원
  [비밀 · 마피아 강태호] 당신의 직업: 마피아
  [비밀 · 마피아 강태호] 동료 마피아: 윤서진
  [비밀 · 마피아 윤서진] 당신의 직업: 마피아
  [비밀 · 마피아 윤서진] 동료 마피아: 강태호
  [비밀 · 경찰 박준수] 당신의 직업: 경찰
  [비밀 · 시민 최민지] 당신의 직업: 시민
  [비밀 · 의사 정도현] 당신의 직업: 의사
  [비밀 · 시민 한소율] 당신의 직업: 시민
  [비밀 · 시민 오지환] 당신의 직업: 시민
  [비밀 · 시민 임채원] 당신의 직업: 시민

===== 1일차 밤 =====
    ┌─ 사고 과정 · 마피아 강태호 · 밤 처치
    │ 받은 정보(PlayerView): 생존 8명 · 공개 기록 0줄 · 지난 속마음 0개 · 후보 6명
    │ 나만 아는 정보: 동료 마피아: 윤서진
    │ RAG 검색: p.5 마피아 전략 가이드 (role=mafia 필터)
    │ 도구 호출: choose_kill_target(target='박준수')
    │   └ 도구 응답: 박준수 선택 완료. 한 줄로 마무리하세요.
    │ 이유: 박준수는 경찰/의사로 보이는 사람으로, 논리적으로 추리해 주목받는 사람으로 판단되며, 시민팀에 위협이 될 수 있는 인물입니다.
    └─ 결정: 박준수 (에이전트가 도구를 호출해 결정)
  [비밀 · 마피아 강태호] 박준수를 노립니다 (이유: 박준수는 경찰/의사로 보이는 사람으로, 논리적으로 추리해 주목받는 사람으로 판단되며, 시민팀에 위협이 될 수 있는 인물입니다.) [제안]
    ┌─ 사고 과정 · 마피아 윤서진 · 

## 4. 참여 모드: 사람 1 + AI 7

- 사람의 이름은 `config.HUMAN_NAME`(기본 "서지우", 본인 이름으로 바꿔도 됨), AI는 캐릭터 8명 중 7명이 무작위로 참가한다.
- 입력 방법: **번호 또는 이름**을 입력한다. 발언은 자유롭게 문장을 입력한다 (빈 입력은 패스).
- `?질문`: 사회자(룰북 RAG, `type=rule` 필터)에게 규칙을 물어본다. 예: `?경찰은 누구를 조사할 수 있나요`
- 사람이 처치/보호/조사 대상을 정하는 밤에는 본인의 비공개 정보만 화면에 나온다.

> 실행하면 입력창이 뜬다. 입력을 하지 않고 넘어가려면 이 셀은 건너뛰어도 된다.

In [7]:
# ===== 6. 참여 모드 실행 =====
HUMAN_SEED = None   # 숫자를 넣으면 같은 배정으로 재현된다

ai_names = random.Random(HUMAN_SEED).sample(config.CHARACTERS, 7)
h_state = core.new_game(ai_names, config.HUMAN_NAME, seed=HUMAN_SEED)
h_controllers = agents.build_controllers(h_state, vs)   # 사람은 HumanController (input 사용)
h_engine = core.GameEngine(h_state, h_controllers)

h_winner = h_engine.run()
print("\n승자:", h_winner)


===== 마피아 게임 시작 (8명) =====
참가자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
  [비밀 · 경찰 서지우] 당신의 직업: 경찰

===== 1일차 밤 =====
----------------------------------------
[1일차 night] 당신은 경찰 입니다.
생존자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
조사할 대상을 고르세요.
  1. 정도현
  2. 박준수
  3. 강태호
  4. 한소율
  5. 윤서진
  6. 오지환
  7. 최민지
번호 또는 이름 (?질문 으로 규칙 문의) > ?의사는 자기 자신을 보호할 수 있나요?   ← (자동 응답)
[사회자] 네, 의사는 자기 자신을 보호할 수 있으며, 연속으로 보호할 수도 있습니다.
(룰북 p.3) (룰북 p.4)
번호 또는 이름 (?질문 으로 규칙 문의) > 4   ← (자동 응답)
밤사이 아무도 사망하지 않았습니다
  [비밀 · 경찰 서지우] 한소율 조사 결과: 마피아다

===== 1일차 낮 토론 =====
[윤서진] 현재까지의 발언은 없으므로 논리적으로 추리하겠습니다. 의심할 대상은 아직 없습니다. 근거가 부족합니다. 첫 낮은 정보가 적으니 특이한 사람을 찾겠습니다.
[오지환] 윤서진아! 왜 아무도 의심 안 했는데 그런 말을 했어? 느낌이 와! 진짜로!
[한소율] 오지환 씨, 혹시 윤서진 씨가 아무도 의심하지 않았다고 말했을 때, 왜 그런 반응을 보이셨나요? 조금 의심돼요. 윤서진 씨는 시민일 수도 있을 텐데, 오지환 씨는 왜 그런 말을 했는지 궁금해요. 다
----------------------------------------
[1일차 day] 당신은 경찰 입니다.
생존자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
[비공개] 1일차 밤 조사 결과: 한소율은 마피아입니다
발언을 입력하세요 (?질문 으로 규칙 문의, 빈 입력은 패스) > 저는 시민입니다. 오지환님은 투표할 때 

In [8]:
# ===== 7. 참여 모드 결과 공개 =====
print(core.reveal_report(h_state))
print("로그:", core.save_log(h_state))

===== 게임 결과: 마피아팀 승리 (6일차) =====

[역할 공개]
  정도현 : 시민 (생존)
  박준수 : 시민 (사망)
  강태호 : 시민 (사망)
  한소율 : 마피아 (생존)
  윤서진 : 마피아 (사망)
  오지환 : 시민 (사망)
  최민지 : 의사 (사망)
  서지우 : 경찰 (사망) [사람]

[밤 기록]
  1일차: 처치 최민지, 보호 최민지, 의사가 살림, 경찰 조사 한소율(마피아)
  2일차: 처치 서지우, 보호 윤서진, 사망 서지우, 경찰 조사 강태호(비마피아)
  3일차: 처치 최민지, 보호 최민지, 의사가 살림
  4일차: 처치 강태호, 보호 강태호, 의사가 살림
  5일차: 처치 최민지, 보호 최민지, 의사가 살림
  6일차: 처치 최민지, 보호 정도현, 사망 최민지

[속마음 요약]
  <정도현 / 시민>
    4일차 day: 강태호가 오지환을 처형했고, 윤서진을 지목한 행동이 마피아 같아. 윤서진이 처형되면 강태호가 마피아일 가능성 높아. 한소율은 강태호를 의심했고,... 의심: 강태호 80, 한소율 60
    4일차 vote: [투표] 강태호: 강태호는 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
    5일차 day: 오지환 처형 때 강태호, 박준수, 정도현, 한소율이 지목했고, 윤서진 처형 때도 강태호, 박준수, 한소율이 지목했어. 강태호가 오지환을 처형하고... 의심: 한소율 70, 최민지 60
    5일차 vote: [투표] 한소율: 한소율은 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
    5일차 vote: [재투표] 한소율: 한소율은 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
  <박준수 / 시민>
    1일차 vote: [투표] 오지환: 오지환은 처음부터 아무도 의심 안 했다고 말했지만, 그 말이 오히려 마피아의 행동을 숨기기 위한 

In [9]:
print("===== 03 완료: 이제 04_langgraph_mafia.ipynb 로 이동하세요 =====")

===== 03 완료: 이제 04_langgraph_mafia.ipynb 로 이동하세요 =====
